# Notebook 3 — Evaluation

Loads each LoRA checkpoint and evaluates on AIME24 / AIME25 / AMC12.
Metrics: `pass@1`, `pass@3`, `acc@3`.

Change `CHECKPOINT` in Cell 2 to switch between ablation variants.
Run Cell 7 after evaluating all variants to print the ablation table.

In [ ]:
# Cell 1 — Setup
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers peft math-verify sympy tqdm

import os
BASE_DIR  = "/content/drive/MyDrive/P-ALIGN"
MODEL_DIR = f"{BASE_DIR}/models/Qwen2.5-1.5B-Instruct"
EVAL_DIR  = f"{BASE_DIR}/eval"
os.makedirs(EVAL_DIR, exist_ok=True)

# Path to the repo's data/result folder (relative to Drive mount)
# Adjust if you copied these files to Drive under a different path.
RESULTS_DIR = f"{BASE_DIR}/../data/result"

In [ ]:
# Cell 2 — Config
# ── CHANGE THIS FOR EACH CHECKPOINT ────────────────────────────────────────
CHECKPOINT = f"{BASE_DIR}/checkpoints/sft_L2"
# ───────────────────────────────────────────────────────────────────────────
N_SAMPLES  = 3     # continuations per question for pass@N
TEMP       = 0.6
TOP_P      = 0.9
MAX_NEW    = 4096
print(f"Evaluating: {CHECKPOINT}")

In [ ]:
# Cell 3 — Load Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer  = AutoTokenizer.from_pretrained(MODEL_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.float16, device_map="auto"
)
model = PeftModel.from_pretrained(base_model, CHECKPOINT)
model.eval()
print(f"Model on {next(model.parameters()).device}")

In [ ]:
# Cell 4 — Load Eval Benchmarks
import json

def load_benchmark(path: str) -> list:
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

benchmarks = {
    "AIME24": load_benchmark(f"{RESULTS_DIR}/aime24-output.jsonl"),
    "AIME25": load_benchmark(f"{RESULTS_DIR}/aime25-output.jsonl"),
    "AMC12":  load_benchmark(f"{RESULTS_DIR}/amc12-output.jsonl"),
}
for name, rows in benchmarks.items():
    print(f"{name}: {len(rows)} questions")

In [ ]:
# Cell 5 — Inference
from tqdm import tqdm

def generate_answers(question: str, n: int = N_SAMPLES) -> list:
    prompt = (
        "Solve the following math problem step by step. "
        "Put your final answer within \\boxed{}.\n"
        f"Problem: {question}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs_list = []
    for _ in range(n):
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=MAX_NEW,
                do_sample=True, temperature=TEMP, top_p=TOP_P,
                pad_token_id=tokenizer.eos_token_id,
            )
        outputs_list.append(
            tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        )
    return outputs_list

ckpt_name = os.path.basename(CHECKPOINT)
results = {}
for bench_name, bench_rows in benchmarks.items():
    out_path = f"{EVAL_DIR}/{ckpt_name}_{bench_name}.jsonl"
    if os.path.exists(out_path):
        print(f"[Skip] {out_path} exists.")
        results[bench_name] = out_path
        continue
    with open(out_path, "w") as fout:
        for row in tqdm(bench_rows, desc=bench_name):
            row["outputs"] = generate_answers(row["question"])
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
    results[bench_name] = out_path
    print(f"{bench_name} done → {out_path}")

In [ ]:
# Cell 6 — Evaluate
import signal, re
from math_verify import parse, verify

def extract_boxed(text: str) -> str:
    m = re.findall(r"\\boxed\{([^}]*)\}", text)
    return m[-1] if m else ""

def score_outputs(outputs: list, answer: str) -> list:
    labels = []
    for pred in outputs:
        try:
            p = parse(extract_boxed(pred))
            g = parse("$" + str(answer) + "$")
            labels.append(int(bool(verify(g, p))))
        except Exception:
            labels.append(0)
    return labels

def compute_metrics(result_file: str) -> dict:
    with open(result_file) as f:
        rows = [json.loads(l) for l in f]
    pass1_list, passn_list, acc_list = [], [], []
    for row in rows:
        labels = score_outputs(row["outputs"], row["answer"])
        pass1_list.append(labels[0])
        passn_list.append(int(any(labels)))
        acc_list.append(sum(labels) / len(labels))
    return {
        "n":                       len(rows),
        "pass@1":                  sum(pass1_list) / len(pass1_list),
        f"pass@{N_SAMPLES}": sum(passn_list) / len(passn_list),
        f"acc@{N_SAMPLES}":  sum(acc_list)   / len(acc_list),
    }

for bench_name, fpath in results.items():
    m = compute_metrics(fpath)
    print(f"\n{'='*40}")
    print(f"  {bench_name}  —  {ckpt_name}")
    for k, v in m.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Cell 7 — Ablation Table
# Fill after running all variants: baseline, L1, L2, L2_pf

table = {
    "P-ALIGN (baseline, K=1)": {"AIME24 pass@1": "?", "AIME25 pass@1": "?", "AMC12 pass@1": "?"},
    "L1 (K=4, soft filter)":   {"AIME24 pass@1": "?", "AIME25 pass@1": "?", "AMC12 pass@1": "?"},
    "L2 (K=4, adv-weight)":    {"AIME24 pass@1": "?", "AIME25 pass@1": "?", "AMC12 pass@1": "?"},
    "L2 + prefix-feedback":    {"AIME24 pass@1": "?", "AIME25 pass@1": "?", "AMC12 pass@1": "?"},
}

header = ["Method"] + list(list(table.values())[0].keys())
print("| " + " | ".join(header) + " |")
print("|" + "---|" * len(header))
for method, vals in table.items():
    row = [method] + [str(v) for v in vals.values()]
    print("| " + " | ".join(row) + " |")